# Proyecto Final Parte 3 - Tokenización de MIDI

### Nombre: Roberto Alejandro Chagra Martinez
### Codigo: 208447128

## Limpieza, Verificación y Reproducción de dataset de archivos MIDI

###1. Verificación de la Cantidad y Calidad Inicial

Antes de alterar los datos, debemos auditar la utilidad real. En la música, el tamaño del archivo en disco no correlaciona directamente con la cantidad de información útil.

- **Cálculo de la duración real:** Un archivo MIDI de 10 KB puede contener cinco minutos de música densa, mientras que uno de 50 KB podría consistir en una sola nota sostenida llena de automatizaciones de volumen innecesarias. Es fundamental iterar sobre el conjunto utilizando librerías como mido o pretty_midi en Python para extraer y sumar el tiempo total de reproducción.

- **Densidad de eventos y silencios:** Cuenta la cantidad de mensajes de encendido y apagado de notas. Descarta programáticamente los archivos que caigan por debajo de un umbral útil (por ejemplo, menos de 50 notas en total) o aquellos donde más del 70% del tiempo de reproducción consista en silencios prolongados.

- **Identificación de archivos corruptos:** Los repositorios masivos extraídos de internet suelen contener archivos truncados, con encabezados dañados o marcas de tiempo negativas que romperán el entrenamiento. Implementa un script que cargue cada archivo dentro de un bloque de manejo de excepciones y elimine automáticamente los que fallen la validación estructural.

In [37]:
#@title Preparacion: Instalación bibliotecas

# Instalar las bibliotecas de Python
!pip -q install kagglehub pretty_midi torch numpy setuptools midi2audio
!apt-get install -y fluidsynth fluid-soundfont-gm
!apt-get upgrade -y fluidsynth
!pip -q install mido

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fluidsynth is already the newest version (2.3.4-1build3).
fluid-soundfont-gm is already the newest version (3.1-5.3).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fluidsynth is already the newest version (2.3.4-1build3).
Calculating upgrade... Done
The following packages have been kept back:
  libcudnn9-cuda-13 libcudnn9-dev-cuda-13 libcudnn9-headers-cuda-13
  libnccl-dev libnccl2
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [38]:
#@title Preparación: Instalación de librerias

import pretty_midi
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import time
import kagglehub
from midi2audio import FluidSynth
from IPython.display import Audio, display

In [39]:
#@title 0. Rutas

# Descarga todo el dataset
path = kagglehub.dataset_download("hansespinosa2/nin-video-game-midis")

# Como solo quiero utilizar los MIDIa SuperMario3DWorld se indica la ruta donde se descargo el dataset
SuperMario3DWorld_dir = os.path.join(path, "nin_midi_files", "nin_midi_files", "Nintendo", "WiiU", "SuperMario3DWorld")

# Se trabajara directamente con esa carpeta
print("Archivos listos:", os.listdir(SuperMario3DWorld_dir))

archivos_midi = []

ruta_dataset = SuperMario3DWorld_dir

for directorio_actual, subdirectorios, archivos in os.walk(ruta_dataset):
    for archivo in archivos:
        if archivo.endswith(('.mid', '.midi')):
            ruta_completa = os.path.join(directorio_actual, archivo)
            archivos_midi.append(ruta_completa)

if archivos_midi:
    archivo_origen = archivos_midi[0]
    print(f"\nSe encontraron {len(archivos_midi)} archivos MIDI.")
    print(f"El primer archivo es: {archivo_origen}")
else:
    print("\nNo se encontraron archivos MIDI en el dataset.")

Using Colab cache for faster access to the 'nin-video-game-midis' dataset.
Archivos listos: ['ShiftyBooMansion.mid', 'WorldClear.mid', 'FuzzyFloodMine.mid', 'World1.mid', 'WorldStar.mid', 'BowsersLavaLakeKeep.mid', 'World3.mid', 'CaptainToadGoesForth.mid', 'SwitchScrambleCircus.mid', 'CourseClear.mid', 'GettingReady.mid', 'TitleScreen.mid', 'CaptainToadPlaysPeekaBoo.mid', 'FootlightLane.mid', 'FinalCurtainforMeowser.mid', 'GrandFinale.mid', 'SuperBellHill.mid', 'CaptainToadsFieryFinale.mid', 'ChainLinkCharge.mid', 'CourseClearShortVersion.mid', 'SprixieKingdom.mid']

Se encontraron 21 archivos MIDI.
El primer archivo es: /kaggle/input/nin-video-game-midis/nin_midi_files/nin_midi_files/Nintendo/WiiU/SuperMario3DWorld/ShiftyBooMansion.mid


In [40]:
#@title Fase 1: Verificar Calidad

def fase1_verificar_calidad(ruta_dataset, duracion_minima=2.0):
    """
    Carga el archivo MIDI de forma segura y verifica que tenga suficiente
    duración para generar secuencias de entrenamiento útiles.
    """
    try:
        midi = pretty_midi.PrettyMIDI(ruta_dataset)
        duracion = midi.get_end_time()

        if duracion < duracion_minima:
            print(f"[Fase 1] Descartado por longitud: {ruta_dataset} ({duracion:.2f}s)")
            return None

        return midi

    except Exception as e:
        print(f"[Fase 1] Archivo corrupto o ilegible ({ruta_dataset}): {e}")
        return None

### 2. Limpieza de Estructura y Metadatos
Los archivos MIDI carecen de un estándar único de grabación, proviniendo de interpretaciones en vivo, secuenciadores antiguos o exportaciones de partituras.

- **Estandarización del formato (Tipo 0 vs. Tipo 1):** El MIDI Tipo 0 colapsa toda la información musical en una sola pista mezclada, mientras que el Tipo 1 separa los instrumentos en múltiples pistas. Para extraer características de manera consistente, debes unificar el conjunto de datos hacia un solo formato, mapeando las pistas según el estándar General MIDI para identificar qué canal corresponde a qué instrumento.

- **Filtrado de compases y tiempos:** Mezclar ritmos estructurados en 4/4 con valses en 3/4 o compases irregulares introducirá un alto grado de ruido si el modelo intenta aprender patrones rítmicos globales. Escanea los metadatos de compás y conserva únicamente aquellos archivos que compartan la estructura temporal deseada para tu análisis.

- **Normalización del tempo:** Un mismo patrón melódico se representa con duraciones matemáticas distintas si el archivo está a 80 BPM versus 140 BPM. Puedes reescribir los metadatos para escalar todas las canciones a un tempo uniforme, permitiendo que el análisis se centre en las relaciones armónicas sin el sesgo de la velocidad original.

In [41]:
#@title Fase 2: Limpiar Estructura

def fase2_limpiar_estructura(midi):
    """
    Filtra la estructura interna del MIDI para conservar únicamente
    las pistas melódicas o armónicas, eliminando la percusión.
    """
    instrumentos_validos = []

    for instrumento in midi.instruments:
        # El atributo is_drum identifica internamente si pertenece al canal de percusión
        if not instrumento.is_drum:
            instrumentos_validos.append(instrumento)

    midi.instruments = instrumentos_validos
    return midi

### 3. Limpieza de Contenido Musical
En esta etapa se refina la información para reducir la complejidad del vocabulario que procesarán los algoritmos.

- **Cuantización temporal:** Las interpretaciones humanas grabadas mediante teclados MIDI presentan micro-variaciones (notas que entran milisegundos antes o después del tiempo exacto). La cuantización ajusta matemáticamente el inicio y fin de todas las notas a una cuadrícula rítmica estricta (como corcheas o semicorcheas perfectas). Esto reduce drásticamente el espacio de estados posibles en la dimensión del tiempo.

- **Aplanamiento de dinámicas (Velocity):** La fuerza con la que se presiona cada tecla se registra en un rango de 0 a 127. Para simplificar los datos, puedes agrupar estos valores en rangos discretos (suave, medio, fuerte) o sobrescribirlos con un valor constante (por ejemplo, 90) si la dinámica expresiva no aporta valor a tu hipótesis actual.

- **Aislamiento de información armónica:** Las pistas del canal 10 están reservadas para la percusión y no contienen altura tonal real. Estas deben filtrarse y eliminarse de los conjuntos de datos enfocados en predicción de melodías o acordes, consolidando únicamente los instrumentos de cuerda, viento o teclado.

In [42]:
#@title Fase 3: Limpiar Contenido Musical

def fase3_limpiar_contenido_musical(midi, pasos_por_segundo=10):
    """
    Cuantiza el inicio y fin de todas las notas en todos los instrumentos
    para alinearlos a una cuadrícula de tiempo estricta.
    """
    for instrumento in midi.instruments:
        for nota in instrumento.notes:
            # Redondear matemáticamente al paso más cercano
            nota.start = round(nota.start * pasos_por_segundo) / pasos_por_segundo
            nota.end = round(nota.end * pasos_por_segundo) / pasos_por_segundo

            # Garantizar que ninguna nota quede con duración de cero segundos
            if nota.end <= nota.start:
                nota.end = nota.start + (1.0 / pasos_por_segundo)

    return midi

### 4. Representación Matemática Final
El paso definitivo es convertir las secuencias MIDI limpias en tensores o matrices numéricas asimilables por modelos predictivos.

- **Rollos de Piano (Piano Rolls):** Transforman el archivo en una matriz bidimensional donde un eje es el tiempo discretizado y el otro abarca los 128 tonos posibles. Esta representación dispersa (sparse matrix) es altamente compatible con arquitecturas que procesan matrices espaciales.

- **Serialización por Tokens:** Consiste en traducir la música a un vocabulario similar al texto mediante eventos secuenciales (ejemplo: Inicia_Nota_60, Avanza_100ms, Termina_Nota_60). Esta estructura lineal define el tamaño real del conjunto de datos en cantidad de "tokens", permitiendo gestionar con precisión el tamaño de los lotes de datos y la longitud de las secuencias de entrada.

In [43]:
#@title Fase 4: Extraer Matriz

def fase4_extraer_matriz(midi, fs=5):
    """Convierte el objeto simbólico en una matriz numérica bidimensional

    y la transforma en un Piano Roll estrictamente BINARIO (0.0 o 1.0).
    """
    # get_piano_roll devuelve por defecto (128 tonos, pasos de tiempo)
    matriz_piano = midi.get_piano_roll(fs=fs)

    # Transponemos para priorizar la dimensión temporal (pasos de tiempo, 128 tonos)
    tensor_temporal = matriz_piano.T

    # Si es 0, se queda apagada (0.0). Esto la hace compatible con BCEWithLogitsLoss.
    tensor_binario = (tensor_temporal > 0).astype(np.float32)

    return tensor_binario

### 5. Función de Tokenización *Musical*
Esta sección implementa la tokenización sobre las matrices (Piano Rolls) extraídas en la Parte 4, cumpliendo con la actividad de clase. En lugar de procesar texto, tratamos cada instante de tiempo musical como una "palabra" (o acorde) compuesta por las notas que suenan simultáneamente.

In [44]:
# @title Fase 5: Función de Tokenización Musical
import numpy as np

def fase4_5_tokenizar_matriz(tensor_matriz):
    """
    Convierte la matriz binaria del Piano Roll en una secuencia de tokens (IDs).
    Agrupa las notas que suenan al mismo tiempo en un solo 'token_str'.
    """
    secuencia_tokens_str = []

    # 1. Convertir cada instante de tiempo (fila) en una "palabra" (string)
    for paso in tensor_matriz:
        # Encontramos qué notas (índices de columna) tienen un 1 (están sonando)
        notas_encendidas = np.where(paso > 0)[0]

        if len(notas_encendidas) > 0:
            # Si hay notas, las unimos con un guion bajo. Ej: "60_64_67"
            token_str = "_".join(map(str, notas_encendidas))
        else:
            # Si no hay notas sonando, lo marcamos como silencio
            token_str = "silencio"

        secuencia_tokens_str.append(token_str)

    # 2. Crear el vocabulario único ordenado
    palabras_unicas = sorted(list(set(secuencia_tokens_str)))
    vocabulario = {palabra: idx + 1 for idx, palabra in enumerate(palabras_unicas)}

    # 3. Convertir la secuencia de strings a IDs numéricos
    ids = [vocabulario[palabra] for palabra in secuencia_tokens_str]

    return ids, vocabulario, secuencia_tokens_str

#### Prueba de Tokenización
En la siguiente celda, tomamos el primer archivo del dataset, lo pasamos por las fases previas de limpieza y extracción, y finalmente aplicamos nuestra nueva función de tokenización para observar los resultados.

In [45]:
# @title Ejecución y Prueba de Tokenización

import os

try:
    # Buscar el primer archivo MIDI disponible
    archivos = [f for f in os.listdir(ruta_dataset) if f.endswith(('.mid', '.midi'))]

    if archivos:
        archivo_prueba = os.path.join(ruta_dataset, archivos[0])
        print(f"Procesando archivo para prueba: {archivos[0]}\n")

        # 1. Pasar por las fases previas (limpieza y extracción de matriz)
        midi = fase1_verificar_calidad(archivo_prueba, duracion_minima=2.0)

        if midi is not None:
            midi = fase2_limpiar_estructura(midi)

            # --- AQUÍ ESTÁ LA CORRECCIÓN ---
            midi = fase3_limpiar_contenido_musical(midi, pasos_por_segundo=5)

            # Extraemos la matriz (esto es lo que antes le pasábamos a la IA)
            matriz = fase4_extraer_matriz(midi, fs=5)

            # 2. Aplicar la Tokenización (El objetivo de esta actividad)
            ids, vocab, tokens_texto = fase4_5_tokenizar_matriz(matriz)

            # 3. Mostrar resultados
            print("--- RESULTADOS DE LA TOKENIZACIÓN ---")
            print(f"Tamaño del vocabulario (acordes/notas únicas): {len(vocab)}")
            print(f"Total de tokens en la canción: {len(ids)}")

            print("\nPrimeros 15 tokens (en formato texto):")
            print(tokens_texto[:15])

            print("\nPrimeros 15 IDs (los números que usaría el modelo):")
            print(ids[:15])

            # (Opcional) Mostrar un ejemplo del diccionario
            print("\nEjemplo de elementos en el vocabulario:")
            items_muestra = list(vocab.items())[:5]
            for clave, valor in items_muestra:
                print(f"  Token: '{clave}' -> ID: {valor}")

        else:
            print("El archivo no pasó la verificación de calidad (Fase 1).")
    else:
        print("No se encontraron archivos MIDI en la ruta especificada.")

except NameError as e:
    print(f"Error: {e}")
    print("Asegúrate de haber ejecutado las celdas de la Parte 1 que definen 'ruta_dataset' y las funciones fase1 a fase4.")

Procesando archivo para prueba: ShiftyBooMansion.mid

--- RESULTADOS DE LA TOKENIZACIÓN ---
Tamaño del vocabulario (acordes/notas únicas): 356
Total de tokens en la canción: 965

Primeros 15 tokens (en formato texto):
['44_87', '44_80_87', '44_80_87', '44_80_87', '44_51_80_87', '44_51_52_80_87', '44_51_52_80_87', '44', '44_50_56_88', '44_50_56_80_88', '44_50_56_80_88', '44_50_56_80_88', '44_50_51_56_80_88', '44_47_50_51_56_80_88', '44_47_50_51_56_80_88']

Primeros 15 IDs (los números que usaría el modelo):
[235, 234, 234, 234, 208, 190, 190, 131, 176, 175, 175, 175, 166, 140, 140]

Ejemplo de elementos en el vocabulario:
  Token: '28_32_40_47_52_59_63_71_75_78_79_80' -> ID: 1
  Token: '28_32_40_47_59_63_71_75_79_80' -> ID: 2
  Token: '28_32_40_59_63_71_75_79_80' -> ID: 3
  Token: '28_32_40_59_63_71_75_80' -> ID: 4
  Token: '28_40_44_51_52_59_63_71_82_83' -> ID: 5


### 6. Segmentación de Secuencias (Ventanas Deslizantes)
Para el entrenamiento supervisado secuencial, no se introduce la canción completa de una vez. Se extraen fragmentos (ventanas) que representan el contexto (X) y se emparejan con el paso temporal inmediatamente posterior que el modelo debe aprender a predecir (Y).

In [46]:
#@title Fase 6: Generar Secuencias

def fase5_generar_secuencias(tensor_cancion, longitud_secuencia):
    """
    Desliza una ventana sobre la matriz completa para generar pares de
    entrenamiento (X: contexto, Y: predicción futura).
    """
    secuencias_x = []
    secuencias_y = []

    # Restamos la longitud para no desbordar el índice final de la matriz
    for i in range(0, len(tensor_cancion) - longitud_secuencia):
        # Extraer ventana de contexto temporal
        x = tensor_cancion[i : i + longitud_secuencia]

        # Extraer ventana objetivo desplazada exactamente 1 paso hacia el futuro
        y = tensor_cancion[i + 1 : i + longitud_secuencia + 1]

        secuencias_x.append(x)
        secuencias_y.append(y)

    return secuencias_x, secuencias_y

### 7. Integración Final como Dataset de PyTorch
Aquí se orquestan las 5 fases anteriores. Al mantener las funciones desacopladas, la clase Dataset queda sumamente limpia y se limita estrictamente a gestionar la carga de archivos, acumular las listas y entregar tensores al DataLoader para la GPU.

In [47]:
#@title Fase 7: Inegracion como Dataset

class PipelineDatasetMIDI(Dataset):
    def __init__(self, directorio_midis, longitud_secuencia=100, fs=5, duracion_minima=10.0):
        # Orquesta el flujo completo de ingeniería de datos MIDI para PyTorch.
        self.fs = fs
        self.muestras_x = []
        self.muestras_y = []

        archivos_procesados = 0

        # Iterar y ejecutar el flujo para cada archivo
        for archivo in os.listdir(directorio_midis):
            if archivo.endswith(('.mid', '.midi')):
                ruta = os.path.join(directorio_midis, archivo)

                # Ejecución secuencial de las Fases 1 a 4
                midi = fase1_verificar_calidad(ruta, duracion_minima)
                if midi is None: continue

                midi = fase2_limpiar_estructura(midi)
                midi = fase3_limpiar_contenido_musical(midi, fs)
                tensor_matriz = fase4_extraer_matriz(midi, fs)

                # Ejecución de Fase 5
                x_cancion, y_cancion = fase5_generar_secuencias(tensor_matriz, longitud_secuencia)

                # Acumular secuencias globales
                self.muestras_x.extend(x_cancion)
                self.muestras_y.extend(y_cancion)

                archivos_procesados += 1

        print(f"[Fase 6] Pipeline completado. Archivos ingeridos: {archivos_procesados}")
        print(f"[Fase 6] Total de secuencias (ventanas) generadas: {len(self.muestras_x)}")

    def __len__(self):
        return len(self.muestras_x)

    def __getitem__(self, indice):
        # Convertir a tensor de PyTorch (float32 es requerido por las capas lineales y RNNs)
        x_tensor = torch.tensor(self.muestras_x[indice], dtype=torch.float32)
        y_tensor = torch.tensor(self.muestras_y[indice], dtype=torch.float32)

        return x_tensor, y_tensor

In [48]:
#@title Procesamiento del archivo

class DatasetArchivoMIDI(Dataset):
    def __init__(self, ruta_dataset, longitud_secuencia=100, fs=5, duracion_minima=10.0):
        """
        Orquesta el flujo de ingeniería de datos para un archivo MIDI,
        preparándolo para ser consumido por un DataLoader de PyTorch.
        """
        self.fs = fs
        self.muestras_x = []
        self.muestras_y = []

        # 1. Pasas la ruta directamente a la Fase 1
        midi = fase1_verificar_calidad(ruta_dataset, duracion_minima)

        # 2. Si el archivo pasó la prueba, ejecutas el resto del pipeline
        if midi is not None:
            midi = fase2_limpiar_estructura(midi)
            midi = fase3_limpiar_contenido_musical(midi, fs)
            tensor_matriz = fase4_extraer_matriz(midi, fs)

            x_cancion, y_cancion = fase5_generar_secuencias(tensor_matriz, longitud_secuencia)

            # Acumular las secuencias de esta única canción
            self.muestras_x.extend(x_cancion)
            self.muestras_y.extend(y_cancion)

            print(f"[Fase 6] Pipeline exitoso. Secuencias extraídas: {len(self.muestras_x)}")
        else:
            print("[Fase 6] Proceso detenido: El archivo no pasó la Fase 1.")

    def __len__(self):
        return len(self.muestras_x)

    def __getitem__(self, indice):
        # Convertir a tensores float32 en el momento de la consulta
        x_tensor = torch.tensor(self.muestras_x[indice], dtype=torch.float32)
        y_tensor = torch.tensor(self.muestras_y[indice], dtype=torch.float32)

        return x_tensor, y_tensor

#### Ejecución
Para poner a prueba este código con un archivo de tu computadora o desde un dataset en linea, pasas la ruta en el momento de crear el objeto de la clase:

In [49]:
#@title Ejecucion

class DatasetDirectorioMIDI(Dataset):
    def __init__(self, ruta_dataset, longitud_secuencia=100, fs=5, duracion_minima=10.0):
        # Se inicializa el atributo 'fs' para que esté disponible en la instancia del objeto
        self.fs = fs
        self.muestras_x = []
        self.muestras_y = []

        # 1. Encontrar todos los archivos MIDI
        archivos_midi = []
        for directorio_actual, subdirectorios, archivos in os.walk(ruta_dataset):
            for archivo in archivos:
                if archivo.endswith(('.mid', '.midi')):
                    archivos_midi.append(os.path.join(directorio_actual, archivo))

        print(f"Se encontraron {len(archivos_midi)} archivos MIDI. Iniciando extracción...")

        # 2. Procesar cada archivo
        archivos_exitosos = 0
        for ruta_archivo in archivos_midi:
            midi = fase1_verificar_calidad(ruta_archivo, duracion_minima)

            if midi is not None:
                try:
                    midi = fase2_limpiar_estructura(midi)
                    midi = fase3_limpiar_contenido_musical(midi, self.fs)
                    tensor_matriz = fase4_extraer_matriz(midi, self.fs)

                    x_cancion, y_cancion = fase5_generar_secuencias(tensor_matriz, longitud_secuencia)

                    self.muestras_x.extend(x_cancion)
                    self.muestras_y.extend(y_cancion)
                    archivos_exitosos += 1
                except Exception as e:
                    print(f"Error procesando {ruta_archivo}: {e}")

        print(f"\n[Resumen] Pipeline completado. {archivos_exitosos}/{len(archivos_midi)} archivos procesados.")
        print(f"Total de secuencias extraídas: {len(self.muestras_x)}")

    def __len__(self):
        return len(self.muestras_x)

    def __getitem__(self, indice):
        x_tensor = torch.tensor(self.muestras_x[indice], dtype=torch.float32)
        y_tensor = torch.tensor(self.muestras_y[indice], dtype=torch.float32)
        return x_tensor, y_tensor

## LSTM (Long Short-Term Memory)

In [50]:
#@title Modelo de Red Neuronal LSTM para Generación de Secuencias MIDI

class GeneradorMIDI_LSTM(nn.Module):
    def __init__(self, tamano_entrada=128, tamano_oculto=256, capas=2, dropout=0.2):
        super(GeneradorMIDI_LSTM, self).__init__()

        # El tamaño oculto define la "capacidad de memoria" de la red
        self.tamano_oculto = tamano_oculto
        self.capas = capas

        # 1. Capa LSTM principal
        # batch_first=True es obligatorio porque nuestro tensor empieza con el tamaño de lote (16)
        self.lstm = nn.LSTM(
            input_size=tamano_entrada,
            hidden_size=tamano_oculto,
            num_layers=capas,
            batch_first=True,
            dropout=dropout
        )

        # 2. Capa lineal de salida
        # Mapea los 256 valores internos de la LSTM de vuelta a las 128 notas musicales
        self.capa_lineal = nn.Linear(tamano_oculto, tamano_entrada)

    def forward(self, x):
        # x entra con dimensiones: [Batch, Time, Features] -> [16, 50, 128]

        # Pasar por la LSTM.
        # 'salida_lstm' contiene las características extraídas para CADA paso temporal
        salida_lstm, (estado_oculto, estado_celda) = self.lstm(x)
        # salida_lstm dimensiones: [16, 50, 256]

        # Pasar todas las salidas temporales por la capa lineal
        predicciones = self.capa_lineal(salida_lstm)
        # predicciones dimensiones: [16, 50, 128]

        return predicciones

# Instanciar el modelo
modelo = GeneradorMIDI_LSTM(tamano_entrada=128, tamano_oculto=256, capas=2)
print(modelo)

GeneradorMIDI_LSTM(
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (capa_lineal): Linear(in_features=256, out_features=128, bias=True)
)


### 0. Configuración del Entrenamiento

In [51]:
#@title Configuración Entrenamiento

# 1. Asegurar que el DataLoader esté inicializado de forma robusta y segura sin AttributeErrors
has_correct_fs = False
try:
    if "dataset_midi_completo" in globals() and dataset_midi_completo.fs == 5:
        has_correct_fs = True
except AttributeError:
    try:
        # Si el dataset existe pero no tiene el atributo fs, se lo asignamos dinámicamente
        dataset_midi_completo.fs = 5
        has_correct_fs = True
    except NameError:
        pass

if "dataloader_prueba" not in globals() or not has_correct_fs:
    if "ruta_dataset" in globals():
        dataset_midi_completo = DatasetDirectorioMIDI(
            ruta_dataset=ruta_dataset,
            longitud_secuencia=50,
            fs=5,
            duracion_minima=2.0,
        )
        # Forzamos la asignación para garantizar que exista el atributo
        dataset_midi_completo.fs = 5
        dataloader_prueba = DataLoader(
            dataset_midi_completo, batch_size=16, shuffle=True
        )
    else:
        raise NameError(
            "No se encontró 'ruta_dataset'. Por favor, ejecuta primero la celda de inicialización."
        )

# 2. Resetear el modelo para borrar los pesos corruptos anteriores y definir la función de pérdida y el optimizador
modelo = GeneradorMIDI_LSTM(tamano_entrada=128, tamano_oculto=256, capas=2)
criterio = nn.BCEWithLogitsLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.001)

# 3. Configurar hiperparámetros de entrenamiento
epocas = 35

print(f"Iniciando el entrenamiento por {epocas} épocas...")
modelo.train()

for i_epoca in range(epocas):
    perdida_acumulada = 0.0
    lotes_procesados = 0

    # Recorrer todos los lotes generados por el DataLoader
    for lote_x, lote_y in dataloader_prueba:

        # Limpiar gradientes de la iteración anterior
        optimizador.zero_grad()

        # Pasar los datos por el modelo (Forward pass)
        predicciones = modelo(lote_x)

        # Calcular el error
        perdida = criterio(predicciones, lote_y)

        # Calcular gradientes (Backward pass)
        perdida.backward()

        # Evita el colapso numérico limitando los gradientes
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)

        # Actualizar los pesos de la red
        optimizador.step()

        perdida_acumulada += perdida.item()
        lotes_procesados += 1

    # Calcular y mostrar el progreso
    if lotes_procesados > 0:
        perdida_promedio = perdida_acumulada / lotes_procesados
    else:
        perdida_promedio = 0.0

    # Imprimir progreso cada 10 épocas y en la primera
    # Corregido: i_epoca en lugar de epoca
    if (i_epoca + 1) % 10 == 0 or i_epoca == 0:
        print(f"Época [{i_epoca+1}/{epocas}] - Pérdida promedio: {perdida_promedio:.4f}")

print("\n¡Entrenamiento completado!")

Iniciando el entrenamiento por 35 épocas...
Época [1/35] - Pérdida promedio: 0.0916
Época [10/35] - Pérdida promedio: 0.0126
Época [20/35] - Pérdida promedio: 0.0036
Época [30/35] - Pérdida promedio: 0.0024

¡Entrenamiento completado!


### Generar nueva musica

### 1. Función de Generación Autoregresiva
En esta etapa el modelo opera en modo de evaluación. Tomamos una secuencia inicial, predecimos el siguiente instante de tiempo, lo añadimos al final de nuestra secuencia, descartamos el instante más antiguo (deslizando la ventana) y repetimos el proceso.

In [52]:
#@title Generación Autoregresiva

def generar_musica(modelo, semilla, pasos_a_generar=1200, umbral=0.5):
    """
    Genera una nueva secuencia musical alimentando al modelo de forma recursiva.

    modelo: red neuronal entrenada (LSTM).
    semilla: Tensor inicial de dimensiones [1, Longitud_Secuencia, 128].
    pasos_a_generar: Cantidad de instantes de tiempo a crear.
    umbral: Valor entre 0 y 1 para decidir si una nota se enciende.
    """
    modelo.eval() # Desactivar dropout y estabilizar capas

    # Clonar la semilla para no modificar el original
    secuencia_actual = semilla.clone().detach()
    generacion_final = []

    with torch.no_grad(): # Desactivar el cálculo de gradientes para ahorrar memoria
        for _ in range(pasos_a_generar):
            # El modelo devuelve predicciones (logits) para todos los pasos temporales: [1, seq_len, 128]
            predicciones_logits = modelo(secuencia_actual)

            # Solo nos interesa predecir el futuro inmediato: el ÚLTIMO paso temporal de la secuencia
            ultimo_paso_logits = predicciones_logits[:, -1, :] # Dimensiones: [1, 128]

            # Aplicar Sigmoide para obtener probabilidades (0.0 a 1.0)
            probabilidades = torch.sigmoid(ultimo_paso_logits)

            # Binarizar las probabilidades según el umbral (0 o 1)
            notas_activas = (probabilidades > umbral).float()

            # Guardar el paso generado
            generacion_final.append(notas_activas.cpu().numpy()[0])

            # Preparar la entrada para la siguiente iteración (desplazar la ventana)
            # notas_activas tiene forma [1, 128], necesitamos expandirla a [1, 1, 128] para concatenar
            nuevo_paso_redimensionado = notas_activas.unsqueeze(1)

            # Concatenar descartando el primer elemento de la secuencia actual
            secuencia_actual = torch.cat([secuencia_actual[:, 1:, :], nuevo_paso_redimensionado], dim=1)

    # Devolver la matriz final generada (Pasos generados, 128)
    return np.array(generacion_final)

### 2. Función de Reconstrucción (Tensor a MIDI)
La matriz generada es un mapa bidimensional donde los unos (1.0) representan notas encendidas en milisegundos específicos. Para reconstruir el archivo .mid, debemos iterar sobre esta cuadrícula y detectar cuándo una nota "nace" (pasa de 0 a 1) y cuándo "muere" (pasa de 1 a 0) para calcular su duración exacta en segundos.

In [53]:
#@title Función Reconstructiva

def tensor_a_midi(matriz_piano, ruta_salida, fs=10, velocidad=90, instrumento=0):
    """
    Convierte un Piano Roll binario de vuelta a un archivo .mid.

    matriz_piano: Array de NumPy de dimensiones (Pasos_Temporales, 128).
    fs: Frecuencia de muestreo usada en la limpieza (ej. 10 pasos por segundo).
    velocidad: Fuerza con la que se tocarán las notas (0-127).
    instrumento: Código de instrumento General MIDI (0 = Acoustic Grand Piano).
    """
    nuevo_midi = pretty_midi.PrettyMIDI()
    programa_instrumento = pretty_midi.instrument_name_to_program(
        pretty_midi.program_to_instrument_name(instrumento)
    )
    pista = pretty_midi.Instrument(program=programa_instrumento)

    # Variables para rastrear qué notas están sonando
    notas_activas = {}

    # Calcular cuánto tiempo real equivale a un paso en la matriz
    duracion_por_paso = 1.0 / fs

    # Recorrer el tiempo (filas) y los 128 tonos posibles (columnas)
    for tiempo_idx, paso in enumerate(matriz_piano):
        tiempo_actual_segundos = tiempo_idx * duracion_por_paso

        for tono in range(128):
            esta_sonando_ahora = paso[tono] > 0
            estaba_sonando_antes = tono in notas_activas

            # Detectar el INICIO de una nota
            if esta_sonando_ahora and not estaba_sonando_antes:
                notas_activas[tono] = tiempo_actual_segundos

            # Detectar el FIN de una nota
            elif not esta_sonando_ahora and estaba_sonando_antes:
                tiempo_inicio = notas_activas.pop(tono)

                # Crear el objeto nota simbólico
                nueva_nota = pretty_midi.Note(
                    velocity=velocidad,
                    pitch=tono,
                    start=tiempo_inicio,
                    end=tiempo_actual_segundos
                )
                pista.notes.append(nueva_nota)

        # Manejar las notas que quedaron "encendidas" al final de la matriz
        tiempo_final = len(matriz_piano) * duracion_por_paso
        for tono, tiempo_inicio in notas_activas.items():
            nueva_nota = pretty_midi.Note(
                velocity=velocidad,
                pitch=tono,
                start=tiempo_inicio,
                end=tiempo_final
            )
            pista.notes.append(nueva_nota)

    nuevo_midi.instruments.append(pista)
    nuevo_midi.write(ruta_salida)
    print(f"Archivo MIDI generado exitosamente en: {ruta_salida}")

### 3. Ejecución Completa
Para unir ambas funciones y escuchar el resultado del modelo, el flujo es el siguiente:

In [59]:
#@title Ejecución Completa con Selector de Instrumento
import ipywidgets as widgets
from IPython.display import display
import os
import torch
import numpy as np
from torch.utils.data import DataLoader

# 1. Lista completa de los 128 instrumentos del estándar General MIDI
lista_instrumentos = [
    ("0 - Acoustic Grand Piano", 0), ("1 - Bright Acoustic Piano", 1), ("2 - Electric Grand Piano", 2),
    ("3 - Honky-tonk Piano", 3), ("4 - Electric Piano 1", 4), ("5 - Electric Piano 2", 5),
    ("6 - Harpsichord", 6), ("7 - Clavinet", 7), ("8 - Celesta", 8), ("9 - Glockenspiel", 9),
    ("10 - Music Box", 10), ("11 - Vibraphone", 11), ("12 - Marimba", 12), ("13 - Xylophone", 13),
    ("14 - Tubular Bells", 14), ("15 - Dulcimer", 15), ("16 - Drawbar Organ", 16),
    ("17 - Percussive Organ", 17), ("18 - Rock Organ", 18), ("19 - Church Organ", 19),
    ("20 - Reed Organ", 20), ("21 - Accordion", 21), ("22 - Harmonica", 22),
    ("23 - Tango Accordion", 23), ("24 - Acoustic Guitar (nylon)", 24), ("25 - Acoustic Guitar (steel)", 25),
    ("26 - Electric Guitar (jazz)", 26), ("27 - Electric Guitar (clean)", 27), ("28 - Electric Guitar (muted)", 28),
    ("29 - Overdriven Guitar", 29), ("30 - Distortion Guitar", 30), ("31 - Guitar Harmonics", 31),
    ("32 - Acoustic Bass", 32), ("33 - Electric Bass (finger)", 33), ("34 - Electric Bass (pick)", 34),
    ("35 - Fretless Bass", 35), ("36 - Slap Bass 1", 36), ("37 - Slap Bass 2", 37),
    ("38 - Synth Bass 1", 38), ("39 - Synth Bass 2", 39), ("40 - Violin", 40),
    ("41 - Viola", 41), ("42 - Cello", 42), ("43 - Contrabass", 43), ("44 - Tremolo Strings", 44),
    ("45 - Pizzicato Strings", 45), ("46 - Orchestral Harp", 46), ("47 - Timpani", 47),
    ("48 - String Ensemble 1", 48), ("49 - String Ensemble 2", 49), ("50 - Synth Strings 1", 50),
    ("51 - Synth Strings 2", 51), ("52 - Choir Aahs", 52), ("53 - Voice Oohs", 53),
    ("54 - Synth Choir", 54), ("55 - Orchestra Hit", 55), ("56 - Trumpet", 56),
    ("57 - Trombone", 57), ("58 - Tuba", 58), ("59 - Muted Trumpet", 59), ("60 - French Horn", 60),
    ("61 - Brass Section", 61), ("62 - Synth Brass 1", 62), ("63 - Synth Brass 2", 63),
    ("64 - Soprano Sax", 64), ("65 - Alto Sax", 65), ("66 - Tenor Sax", 66), ("67 - Baritone Sax", 67),
    ("68 - Oboe", 68), ("69 - English Horn", 69), ("70 - Bassoon", 70), ("71 - Clarinet", 71),
    ("72 - Piccolo", 72), ("73 - Flute", 73), ("74 - Recorder", 74), ("75 - Pan Flute", 75),
    ("76 - Blown bottle", 76), ("77 - Shakuhachi", 77), ("78 - Whistle", 78), ("79 - Ocarina", 79),
    ("80 - Lead 1 (square)", 80), ("81 - Lead 2 (sawtooth)", 81), ("82 - Lead 3 (calliope)", 82),
    ("83 - Lead 4 chiff", 83), ("84 - Lead 5 (charang)", 84), ("85 - Lead 6 (voice)", 85),
    ("86 - Lead 7 (fifths)", 86), ("87 - Lead 8 (bass + lead)", 87), ("88 - Pad 1 (new age)", 88),
    ("89 - Pad 2 (warm)", 89), ("90 - Pad 3 (polysynth)", 90), ("91 - Pad 4 (choir)", 91),
    ("92 - Pad 5 (bowed)", 92), ("93 - Pad 6 (metallic)", 93), ("94 - Pad 7 (halo)", 94),
    ("95 - Pad 8 (sweep)", 95), ("96 - FX 1 (rain)", 96), ("97 - FX 2 (soundtrack)", 97),
    ("98 - FX 3 (crystal)", 98), ("99 - FX 4 (atmosphere)", 99), ("100 - FX 5 (brightness)", 100),
    ("101 - FX 6 (goblins)", 101), ("102 - FX 7 (echoes)", 102), ("103 - FX 8 (sci-fi)", 103),
    ("104 - Sitar", 104), ("105 - Banjo", 105), ("106 - Shamisen", 106), ("107 - Koto", 107),
    ("108 - Kalimba", 108), ("109 - Bagpipe", 109), ("110 - Fiddle", 110), ("111 - Shanai", 111),
    ("112 - Tinkle Bell", 112), ("113 - Agogo", 113), ("114 - Steel Drums", 114),
    ("115 - Woodblock", 115), ("116 - Taiko Drum", 116), ("117 - Melodic Tom", 117),
    ("118 - Synth Drum", 118), ("119 - Reverse Cymbal", 119), ("120 - Guitar Fret Noise", 120),
    ("121 - Breath Noise", 121), ("122 - Seashore", 122), ("123 - Bird Tweet", 123),
    ("124 - Telephone Ring", 124), ("125 - Helicopter", 125), ("126 - Applause", 126),
    ("127 - Gunshot", 127)
]

selector_instrumento = widgets.Dropdown(
    options=lista_instrumentos,
    value=0,
    description='Instrumento:',
    style={'description_width': 'initial'}
)
display(selector_instrumento)

# 2. Botón para iniciar la generación tras seleccionar el instrumento
boton_generar = widgets.Button(description="Generar Música", button_style='primary')
display(boton_generar)

def on_button_clicked(b):
    print(f"\nCargando MIDIs desde: {ruta_dataset}")

    dataset_midi_completo = DatasetDirectorioMIDI(
        ruta_dataset=ruta_dataset,
        longitud_secuencia=50,
        fs=5,
        duracion_minima=2.0
    )

    if len(dataset_midi_completo) > 0:
        dataloader_prueba = DataLoader(dataset_midi_completo, batch_size=16, shuffle=True)
        lote_x, lote_y = next(iter(dataloader_prueba))

        instrumento_elegido = selector_instrumento.value
        nombre_instrumento = [nombre for nombre, val in lista_instrumentos if val == instrumento_elegido][0]

        print(f"\nIniciando generación de nueva música con el instrumento: {nombre_instrumento}...")

        semilla_inicial = lote_x[0:1]
        matriz_generada = generar_musica(modelo, semilla_inicial, pasos_a_generar=600, umbral=0.1)

        # Asegúrate de que la carpeta 'results' exista
        os.makedirs("../results", exist_ok=True)
        ruta_salida = "../results/mi_primera_composicion_ia.mid"

        tensor_a_midi(matriz_generada, ruta_salida, fs=5, velocidad=90, instrumento=instrumento_elegido)
    else:
        print("\nNo se pudieron extraer secuencias de ningún archivo.")

boton_generar.on_click(on_button_clicked)

Dropdown(description='Instrumento:', options=(('0 - Acoustic Grand Piano', 0), ('1 - Bright Acoustic Piano', 1…

Button(button_style='primary', description='Generar Música', style=ButtonStyle())


Cargando MIDIs desde: /kaggle/input/nin-video-game-midis/nin_midi_files/nin_midi_files/Nintendo/WiiU/SuperMario3DWorld
Se encontraron 21 archivos MIDI. Iniciando extracción...

[Resumen] Pipeline completado. 21/21 archivos procesados.
Total de secuencias extraídas: 11251

Iniciando generación de nueva música con el instrumento: 40 - Violin...
Archivo MIDI generado exitosamente en: ../results/mi_primera_composicion_ia.mid

Cargando MIDIs desde: /kaggle/input/nin-video-game-midis/nin_midi_files/nin_midi_files/Nintendo/WiiU/SuperMario3DWorld
Se encontraron 21 archivos MIDI. Iniciando extracción...

[Resumen] Pipeline completado. 21/21 archivos procesados.
Total de secuencias extraídas: 11251

Iniciando generación de nueva música con el instrumento: 10 - Music Box...
Archivo MIDI generado exitosamente en: ../results/mi_primera_composicion_ia.mid


### 4. Reproducción

In [61]:
#@title Reproducir MIDI creado
import os

def reproducir_midi(ruta_midi, ruta_salida_wav="salida_audio.wav"):
    # Ruta del banco de sonidos (SoundFont) estándar de Linux/Ubuntu
    ruta_soundfont = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

    # Verificar que el banco de sonidos exista
    if not os.path.exists(ruta_soundfont):
        print("No se encontró el SoundFont. Asegúrate de instalar fluid-soundfont-gm.")
        return

    # Inicializar el sintetizador con el banco de sonidos
    fs = FluidSynth(ruta_soundfont)

    print(f"Convirtiendo {ruta_midi} a WAV...")

    # Convertir el archivo MIDI a formato de audio WAV
    fs.midi_to_audio(ruta_midi, ruta_salida_wav)

    # Mostrar el reproductor interactivo justo debajo de la celda indicando explícitamente el 'filename'
    display(Audio(filename=ruta_salida_wav))

# Ejecutar
reproducir_midi("../results/mi_primera_composicion_ia.mid")

Convirtiendo ../results/mi_primera_composicion_ia.mid a WAV...
